##  Landelijke dataset maken

In [1]:

import pandas as pd

# =====================================
# 1. ORIGINELE DATASET INLEZEN
# =====================================

df = pd.read_csv(
    "ongevallen.txt",
    sep=",",
    encoding="utf-8-sig",
    low_memory=False
)

print("Original dataset:", df.shape)


# =====================================
# 2. RELEVANTE KOLOMMEN SELECTEREN
# =====================================

columns = [
    "VKL_NUMMER",
    "DATUM_VKL",
    "JAAR_VKL",
    "MND_NUMMER",
    "GME_ID",
    "GME_NAAM",
    "WVK_ID",
    "JTE_ID",
    "MAXSNELHD",
    "AP3_CODE",
    "ANTL_DOD",
    "ANTL_GZH",
    "ANTL_SEH",
    "ANTL_SLA"
]

data = df[columns].copy()


# =====================================
# 3. DATA OPSCHONEN
# =====================================

numeric_cols = [
    "JAAR_VKL",
    "MND_NUMMER",
    "MAXSNELHD",
    "ANTL_DOD",
    "ANTL_GZH",
    "ANTL_SEH",
    "ANTL_SLA"
]

for col in numeric_cols:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

# Tekst opschonen
data["GME_NAAM"] = (
    data["GME_NAAM"]
    .astype("string")
    .str.strip()
)

data["AP3_CODE"] = (
    data["AP3_CODE"]
    .astype("string")
    .str.strip()
    .str.upper()
)

data["WVK_ID"] = (
    data["WVK_ID"]
    .astype("string")
    .str.strip()
)

# Dubbele ongevallen verwijderen
data = data.drop_duplicates(
    subset="VKL_NUMMER"
)


# =====================================
# 4. JAREN FILTEREN: 2016-2025
# =====================================

data = data[
    data["JAAR_VKL"].between(2016, 2025)
].copy()


# =====================================
# 5. AMSTERDAM / REST NEDERLAND
# =====================================

data["area"] = "Rest Netherlands"

data.loc[
    data["GME_NAAM"].str.lower() == "amsterdam",
    "area"
] = "Amsterdam"


# =====================================
# 6. BEFORE / TRANSITION / AFTER
# =====================================

data["period"] = "Before"

data.loc[
    data["JAAR_VKL"] == 2023,
    "period"
] = "Transition"

data.loc[
    data["JAAR_VKL"] >= 2024,
    "period"
] = "After"


# =====================================
# 7. SPEED_CHANGED KOLOM
# =====================================

# Later koppelen met wegvakgegevens
# 1 = van 50 naar 30 km/h
# 0 = onveranderd
# NA = onbekend

data["speed_changed"] = pd.NA


# =====================================
# 8. LANDELIJKE DATASET OPSLAAN
# =====================================

data.to_csv(
    "ongevallen_gefilterd_landelijk.csv",
    index=False
)

print("National dataset:", data.shape)


# =====================================
# 9. AMSTERDAM SELECTEREN
# =====================================

amsterdam = data[
    data["area"] == "Amsterdam"
].copy()

# Area-kolom verwijderen (overbodig)
amsterdam = amsterdam.drop(
    columns=["area"]
)


# =====================================
# 10. AMSTERDAM DATASET OPSLAAN
# =====================================

amsterdam.to_csv(
    "ongevallen_gefilterd_amsterdam.csv",
    index=False
)

print("Amsterdam dataset:", amsterdam.shape)


# =====================================
# 11. CONTROLEREN
# =====================================

print("\nNational data by year:")
print(
    data["JAAR_VKL"]
    .value_counts()
    .sort_index()
)

print("\nAmsterdam data by year:")
print(
    amsterdam["JAAR_VKL"]
    .value_counts()
    .sort_index()
)

print("\nAmsterdam periods:")
print(
    pd.crosstab(
        amsterdam["JAAR_VKL"],
        amsterdam["period"]
    )
)

print("\nFiltering completed!")


Original dataset: (1237603, 66)
National dataset: (1237603, 17)
Amsterdam dataset: (63291, 16)

National data by year:
JAAR_VKL
2016    124992
2017    123930
2018    128225
2019    134617
2020    103578
2021    114219
2022    122036
2023    134005
2024    126380
2025    125621
Name: count, dtype: int64

Amsterdam data by year:
JAAR_VKL
2016    6811
2017    6884
2018    7156
2019    7142
2020    5018
2021    5317
2022    5979
2023    6813
2024    6201
2025    5970
Name: count, dtype: int64

Amsterdam periods:
period    After  Before  Transition
JAAR_VKL                           
2016          0    6811           0
2017          0    6884           0
2018          0    7156           0
2019          0    7142           0
2020          0    5018           0
2021          0    5317           0
2022          0    5979           0
2023          0       0        6813
2024       6201       0           0
2025       5970       0           0

Filtering completed!
